In [1]:
import gradio as gr
from openai import OpenAI
from kittentts import KittenTTS
import soundfile as sf
import numpy as np
import re
import os

c:\Users\GAURAV\miniconda3\envs\multiModal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
tts_model = KittenTTS("KittenML/kitten-tts-nano-0.1")

In [3]:
def stream_text(user_input, history):
    messages = history + [{"role": "user", "content": user_input}]
    full_response = ""
    stream = client.chat.completions.create(model="llama3.2:latest", messages=messages, stream=True)
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            full_response += chunk.choices[0].delta.content
            yield messages + [{"role": "assistant", "content": full_response}], full_response

In [4]:
# With chunking the text so that kittenTTS context window is not exceeded.
def generate_audio(text):
    if not text or len(text.strip()) == 0:
        return None
    
    audio_path = "response_audio.wav"
    MAX_CHARS = 180  # The "safety limit" for KittenTTS Nano's internal buffer
    
    # 1. Initial split by sentence markers (. ! ?)
    sentences = re.split(r'(?<=[.!?]) +', text.replace('\n', ' '))
    
    # 2. Sub-chunking: If a sentence is too long, break it by commas or spaces
    final_chunks = []
    for s in sentences:
        if len(s) <= MAX_CHARS:
            final_chunks.append(s)
        else:
            # Sub-split long sentences by commas or just chunks of words
            words = s.split(' ')
            current_chunk = ""
            for word in words:
                if len(current_chunk) + len(word) < MAX_CHARS:
                    current_chunk += " " + word
                else:
                    final_chunks.append(current_chunk.strip())
                    current_chunk = word
            final_chunks.append(current_chunk.strip())

    combined_audio = []
    
    try:
        for chunk in final_chunks:
            clean_chunk = chunk.strip()
            if len(clean_chunk) < 2:
                continue
    
            chunk_data = tts_model.generate(
                clean_chunk, 
                voice="expr-voice-2-f", 
                speed=1.0
            )
            combined_audio.append(chunk_data)
        
        if not combined_audio:
            return None

        # Stitch the clean chunks together
        final_audio = np.concatenate(combined_audio)
        sf.write(audio_path, final_audio, 24000)
        return audio_path

    except Exception as e:
        print(f"Modular TTS Error: {e}")
        return None

In [5]:
def generate_image(text):
    if not text: return None
    # Imagine your Stable Diffusion logic here
    # image = sd_model.generate(prompt=text)
    return "path_to_generated_image.png"

In [6]:
# --- THE CORRECTED UI LAYOUT ---
with gr.Blocks(fill_height=True, title="Modular AI Studio") as demo:
    # This 'State' acts as a hidden bridge between the Text function and Audio function
    final_text_state = gr.State("")

    gr.HTML("<h1 style='text-align: center; font-family: sans-serif;'>AI Multimodal Studio</h1>")
    
    with gr.Row(equal_height=True):
        # LEFT: Chat Column
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Conversation", height=600)
            with gr.Row():
                msg = gr.Textbox(show_label=False, placeholder="Ask me...", scale=9, container=False)
                submit_btn = gr.Button("Send", variant="primary", scale=1)

        # RIGHT: Media Column
        with gr.Column(scale=1):
            image_viewer = gr.Image(label="Image Output", height=400, interactive=False)
            audio_player = gr.Audio(label="Voice Output", interactive=False, autoplay=True)

    # --- THE CHAIN REACTION LOGIC ---
    
    # STEP 1: When user hits 'Send', run stream_text
    # It updates the chatbot (stream) and the final_text_state (at the very end)
    text_event = submit_btn.click(
        fn=stream_text, 
        inputs=[msg, chatbot], 
        outputs=[chatbot, final_text_state]
    )
    
    # STEP 2: When stream_text finishes, trigger generate_audio
    # It takes the 'final_text_state' as input
    text_event.then(
        fn=generate_audio, 
        inputs=[final_text_state], 
        outputs=[audio_player]
    )
    
    # STEP 3: Trigger generate_image (can run after audio or in parallel)
    text_event.then(
        fn=generate_image, 
        inputs=[final_text_state], 
        outputs=[image_viewer]
    )

    # STEP 4: Clear the input box after everything starts
    text_event.then(lambda: "", None, [msg])

    # Repeat for the 'Enter' key on the Textbox
    msg.submit(
        fn=stream_text, 
        inputs=[msg, chatbot], 
        outputs=[chatbot, final_text_state]
    ).then(
        fn=generate_audio, inputs=[final_text_state], outputs=[audio_player]
    ).then(
        fn=generate_image, inputs=[final_text_state], outputs=[image_viewer]
    ).then(lambda: "", None, [msg])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
